In [1]:
!pip install Faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 21.7 MB/s eta 0:00:00


In [2]:
from faker import Faker
import pandas as pd
import random

In [3]:
fake = Faker()
random.seed(0)
Faker.seed(0)

# -----------------------------
# 1) Courses + prerequisites
# -----------------------------
courses = [
    "Introduction to Computer Science",
    "Structured Programming",
    "Structured Programming Lab",
    "Calculus 1",
    "Calculus 2",
    "Discrete Mathematics",
    "Statistical Methods",
    "Linear Algebra",
    "Object Oriented Programming",
    "Object Oriented Programming Lab",
    "Data Structures and Introduction to Algorithms",
    "Algorithms Design and Analysis",
    "Database Systems",
    "Database Systems Lab",
    "Operating Systems",
    "Operating Systems Lab",
    "Distributed Systems",
    "Computer and Society",
    "Information Systems Security",
    "Introduction to Data Science",
    "Introduction to Data Science Lab",
    "Data Engineering",
    "Data Engineering Lab",
    "Statistics and Probability for Data Science",
    "Artificial Intelligence",
    "Computer Architecture for Machine Learning",
    "Natural Language Processing",
    "High Performance Computing for Big Data",
    "Data Visualization",
    "Pattern Recognition",
    "Data Mining",
    "Business Intelligence"
]

# Prerequisites – note: lab NEVER without its course
prerequisites = {
    "Structured Programming": ["Introduction to Computer Science"],
    "Structured Programming Lab": ["Structured Programming"],
    "Calculus 2": ["Calculus 1"],
    "Discrete Mathematics": ["Calculus 1"],
    "Statistical Methods": ["Calculus 2"],
    "Linear Algebra": ["Calculus 1"],
    "Object Oriented Programming": ["Structured Programming"],
    "Object Oriented Programming Lab": ["Object Oriented Programming"],
    "Data Structures and Introduction to Algorithms": ["Structured Programming"],
    "Algorithms Design and Analysis": ["Data Structures and Introduction to Algorithms"],
    "Database Systems": ["Structured Programming"],
    "Database Systems Lab": ["Database Systems"],
    "Operating Systems": ["Data Structures and Introduction to Algorithms"],
    "Operating Systems Lab": ["Operating Systems"],
    "Distributed Systems": ["Operating Systems"],
    "Information Systems Security": ["Operating Systems"],
    "Introduction to Data Science": [
        "Introduction to Computer Science",
        "Statistical Methods"
    ],
    "Introduction to Data Science Lab": ["Introduction to Data Science"],
    "Statistics and Probability for Data Science": ["Calculus 2"],
    "Data Engineering": ["Introduction to Data Science", "Database Systems"],
    "Data Engineering Lab": ["Data Engineering"],
    "Artificial Intelligence": [
        "Data Structures and Introduction to Algorithms",
        "Statistics and Probability for Data Science"
    ],
    "Computer Architecture for Machine Learning": [
        "Data Structures and Introduction to Algorithms"
    ],
    "Natural Language Processing": ["Artificial Intelligence"],
    "High Performance Computing for Big Data": ["Data Engineering"],
    "Data Visualization": ["Introduction to Data Science"],
    "Pattern Recognition": [
        "Statistics and Probability for Data Science",
        "Linear Algebra"
    ],
    "Data Mining": [
        "Introduction to Data Science",
        "Statistics and Probability for Data Science"
    ],
    "Business Intelligence": ["Data Mining"]
}

# Core / early courses that almost every DS student should take
core_courses = [
    "Introduction to Computer Science",
    "Structured Programming",
    "Calculus 1",
    "Calculus 2",
    "Statistics and Probability for Data Science",
    "Introduction to Data Science",
    "Data Structures and Introduction to Algorithms"
]

In [4]:
# -----------------------------
# 2) Helper: generate student IDs
#    SAFE & FAST (no infinite loops)
#    format: YYYY0XXX  (e.g., 20220110)
# -----------------------------

# Pre-generate ALL possible IDs once


In [5]:
student_ids = []

for year in range(2020, 2025):      # 2020, 2021, 2022, 2023, 2024
    for i in range(1, 1001):        # 0001 .. 1000
        sid = int(f"{year}{i:04d}") # e.g. 2020 + "0001" -> 20200001
        student_ids.append(sid)

num_students = len(student_ids)

In [14]:
# -----------------------------
# 3) Generate data (Core + Big Data + 90% labs concurrency)
# -----------------------------
num_students = 5000

students = []        # main student info table
grades_rows = []     # marks per course
attendance_rows = [] # absences per course

# Core / early courses that almost every DS student should take
core_courses = [
    "Introduction to Computer Science",
    "Structured Programming",
    "Calculus 1",
    "Calculus 2",
    "Statistics and Probability for Data Science",
    "Introduction to Data Science",
    "Data Structures and Introduction to Algorithms"
]

bigdata_course = "High Performance Computing for Big Data"  # must match your courses list

# course → lab mapping (for concurrency)
course_lab_pairs = {
    "Structured Programming": "Structured Programming Lab",
    "Object Oriented Programming": "Object Oriented Programming Lab",
    "Database Systems": "Database Systems Lab",
    "Operating Systems": "Operating Systems Lab",
    "Introduction to Data Science": "Introduction to Data Science Lab",
    "Data Engineering": "Data Engineering Lab"
}

for i in range(num_students):

    # ---- student basic info ----
    sid = student_ids[i]            # uses pre-generated IDs from part 2
    fake=Faker('ar_SA')
    full_name = fake.first_name()+' '+fake.last_name()


    # how many courses this student has completed in total
    target_courses = random.randint(12, 25)

    selected = set()

    # 1) Force core courses first (in order), respecting prerequisites
    for c in core_courses:
        if c not in courses:
            continue  # skip if not in your course list
        prereqs = prerequisites.get(c, [])
        if all(p in selected for p in prereqs):
            selected.add(c)

    # 2) Fill up remaining courses randomly, respecting prerequisites
    remaining_courses = [c for c in courses if c not in selected]

    for course in random.sample(remaining_courses, len(remaining_courses)):
        if len(selected) >= target_courses:
            break
        prereqs = prerequisites.get(course, [])
        if all(p in selected for p in prereqs):
            selected.add(course)

    # 3) Enforce ~90% concurrency for course–lab pairs
    for course, lab in course_lab_pairs.items():
        if course in selected and lab in courses and lab not in selected:
            # about 90% of students who take the course will also have the lab
            if len(selected) < target_courses and random.random() < 0.9:
                selected.add(lab)

    # 4) Explicitly give some students Big Data if they have its prerequisites
    if bigdata_course in courses:
        bigdata_prereqs = prerequisites.get(bigdata_course, [])
        if all(p in selected for p in bigdata_prereqs):
            # e.g. 30% of eligible students actually take it
            if random.random() < 0.3:
                selected.add(bigdata_course)

    # ---- marks + attendance for completed courses ----
    marks = {}
    absences = {}

    for c in selected:
        marks[c] = random.randint(40, 100)   # integer marks
        absences[c] = random.randint(0, 8)   # integer absences

    # GPA out of 100 = average of completed course marks (1 decimal)
    if marks:
        gpa = round(sum(marks.values()) / len(marks), 1)
    else:
        gpa = None

    # main student row (NO course columns here)
    students.append({
        "student_id": sid,
        "full_name": full_name,
        "gpa": gpa
    })

    # grades row: one column per course
    grade_row = {"student_id": sid}
    # attendance row: one column per course
    att_row = {"student_id": sid}

    for c in courses:
        grade_row[c] = marks.get(c, None)
        att_row[c] = absences.get(c, None)

    grades_rows.append(grade_row)
    attendance_rows.append(att_row)

In [15]:
# -----------------------------
# 4) Build DataFrames
# -----------------------------


students_df   = pd.DataFrame(students)
grades_df     = pd.DataFrame(grades_rows)
attendance_df = pd.DataFrame(attendance_rows)

students_df = pd.DataFrame(students).sort_values("student_id")
grades_df = pd.DataFrame(grades_rows).sort_values("student_id")
attendance_df = pd.DataFrame(attendance_rows).sort_values("student_id")

# Reset index after sorting
students_df.reset_index(drop=True, inplace=True)
grades_df.reset_index(drop=True, inplace=True)
attendance_df.reset_index(drop=True, inplace=True)
# Quick check
print("Students table:")
display(students_df.head())

print("\nGrades table:")
display(grades_df.head())

print("\nAttendance table:")
display(attendance_df.head())

Students table:


,student_id,full_name,gpa
0,20200001,موفّق آل بن ظافر,70.7
1,20200002,هند آل علي,74.6
2,20200003,داهي المشاولة,68.0
3,20200004,جوين العليان,69.2
4,20200005,عماد العجلان,70.4



Grades table:


,student_id,Introduction to Computer Science,Structured Programming,Structured Programming Lab,Calculus 1,Calculus 2,Discrete Mathematics,Statistical Methods,Linear Algebra,Object Oriented Programming,...,Data Engineering Lab,Statistics and Probability for Data Science,Artificial Intelligence,Computer Architecture for Machine Learning,Natural Language Processing,High Performance Computing for Big Data,Data Visualization,Pattern Recognition,Data Mining,Business Intelligence
0,20200001,92,67,50.0,67,92,NaN,83.0,NaN,61.0,...,NaN,72,69.0,50.0,NaN,NaN,NaN,NaN,76.0,51.0
1,20200002,67,41,78.0,58,82,90.0,80.0,77.0,NaN,...,NaN,88,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20200003,93,98,56.0,54,87,72.0,58.0,87.0,53.0,...,NaN,43,88.0,55.0,NaN,NaN,NaN,40.0,NaN,NaN
3,20200004,46,75,72.0,81,75,88.0,99.0,49.0,47.0,...,NaN,90,NaN,NaN,NaN,NaN,NaN,56.0,NaN,NaN
4,20200005,91,63,87.0,60,90,NaN,83.0,NaN,NaN,...,NaN,60,NaN,56.0,NaN,NaN,95.0,NaN,NaN,NaN



Attendance table:


,student_id,Introduction to Computer Science,Structured Programming,Structured Programming Lab,Calculus 1,Calculus 2,Discrete Mathematics,Statistical Methods,Linear Algebra,Object Oriented Programming,...,Data Engineering Lab,Statistics and Probability for Data Science,Artificial Intelligence,Computer Architecture for Machine Learning,Natural Language Processing,High Performance Computing for Big Data,Data Visualization,Pattern Recognition,Data Mining,Business Intelligence
0,20200001,1,3,8.0,7,5,NaN,1.0,NaN,3.0,...,NaN,3,2.0,2.0,NaN,NaN,NaN,NaN,4.0,1.0
1,20200002,8,8,1.0,6,8,6.0,3.0,5.0,NaN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20200003,3,0,6.0,1,5,3.0,2.0,4.0,3.0,...,NaN,2,3.0,2.0,NaN,NaN,NaN,8.0,NaN,NaN
3,20200004,7,2,1.0,5,5,7.0,0.0,2.0,0.0,...,NaN,6,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN
4,20200005,4,0,5.0,7,3,NaN,1.0,NaN,NaN,...,NaN,8,NaN,0.0,NaN,NaN,2.0,NaN,NaN,NaN


In [10]:
display(students_df.head())


,student_id,full_name,gpa
0,20200001,"(محسن, آل بن ظافر)",71.6
1,20200002,"(عبد الباقي, المشاولة)",77.0
2,20200003,"(مُتولي, الخرافي)",65.3
3,20200004,"(عذب, العقيل)",68.9
4,20200005,"(فرحان, بن لادن)",73.7


In [9]:
# -----------------------------
# 5) Save to CSV files
# -----------------------------
students_df.to_csv("students_table.csv", index=False)
grades_df.to_csv("grades_table.csv", index=False)
attendance_df.to_csv("attendance_table.csv", index=False)

print("\nCSV files saved:")
print(" - students_table.csv")
print(" - grades_table.csv")
print(" - attendance_table.csv")


CSV files saved:
 - students_table.csv
 - grades_table.csv
 - attendance_table.csv
